In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn.linear_model as lm
import pandas as pd

import mat73

In [ ]:
my_dict = mat73.loadmat('SingleRegion_Aggression_data.mat')

In [ ]:
TrainsetMouse = my_dict['TrainsetMouse']
mbeh_all2 = my_dict['mbeh_all2']
mcond_all2 = my_dict['mcond_all2']
mouse_all2 = my_dict['mouse_all2']
mpow_all2 = my_dict['mpow_all2']
mpow_all3s = my_dict['mpow_all3s']
mtimecondbeh2 = my_dict['mtimecondbeh2']
mu_all3 = my_dict['mu_all3']
testsetMouse = my_dict['testsetMouse']


### Get index of training set

In [ ]:
N_samples = len(mouse_all2)

mice_all = []
for mouse in TrainsetMouse:
    mice_all.append(mouse[0])
for mouse in testsetMouse:
    mice_all.append(mouse[0])
mice_all = np.array(mice_all)

trainset = []
for i in range(len(TrainsetMouse)):
    trainset.append(TrainsetMouse[i][0])

train_idxs = np.zeros(N_samples)
mouse_idxs = np.zeros(N_samples)
for i in range(N_samples):
    if mouse_all2[i][0] == 'Mouse048':
        train_idxs[i] = -1
        continue
    if mouse_all2[i][0] in trainset:
        train_idxs[i] = 1
    mouse_idxs[i] = np.where(mice_all==mouse_all2[i][0])[0][0]
np.mean(train_idxs)

### Get positive and negative indexes of conditions

Positive: can't really evaluate time
Negative

In [ ]:
idxs_pos = (mtimecondbeh2[:,0]<240)&(mtimecondbeh2[:,1]==0)
idx_neg = ((mcond_all2==4)&(mbeh_all2>0))|((mcond_all2==6)&(mbeh_all2==2))|((mcond_all2==8)&(mbeh_all2==2))
selection_indices = idxs_pos|idx_neg
print(np.mean(idxs_pos))
print(np.mean(idx_neg))
print(np.mean(selection_indices))

### Create task labels

In [ ]:
y = np.zeros(N_samples)
y[idxs_pos] = 1

In [ ]:
mpower_reduced = mpow_all2[:,:,selection_indices]
y_reduced = y[selection_indices]
train_idxs_reduced = train_idxs[selection_indices]

In [ ]:
y_train = y_reduced[train_idxs_reduced==1]
y_test = y_reduced[train_idxs_reduced==0]

In [ ]:
m_idx_unique_test = np.unique(mouse_idxs[train_idxs==0])
mouse_idxs_reduced = mouse_idxs[selection_indices]
m_test = mouse_idxs_reduced[train_idxs_reduced==0]

## Now fit all of the individual models

In [ ]:
import sys
sys.path.append('/home/austin/Aggression/Code/NMF')
from nmf_elastic_noprint import NMF_logistic
from sklearn.metrics import roc_auc_score

In [ ]:
nFact = 4
nIter = 20000
mu = 1.0
model_list = []
test_aucs = np.zeros(11)

test_aucs_mouse = np.zeros((11,9))
test_aucs_mouse_notTrue = np.zeros((11,9))

for i in range(11):
    XT = np.squeeze(mpower_reduced[:,i,:])
    X = np.transpose(XT)
    X = X*10
    X[X>6] = 6
    Xtrain = X[train_idxs_reduced==1]
    Xtest = X[train_idxs_reduced==0]
    model = NMF_logistic(nFact,nIter=nIter,LR=1e-3,mu=mu,batchSize=100)
    S_train = model.fit_transform(Xtrain,y_train)
    S_test = model.transform(Xtest)
    test_aucs[i] = roc_auc_score(y_test,np.squeeze(S_test[:,0]*model.Phi))
    model_list.append(model)
    for j in range(9):
        pred_true = np.squeeze(S_test[m_test==20+j,0]*model.Phi)
        pred_standard = np.squeeze(S_test[m_test==20+j,0])
        test_aucs_mouse[i,j] = roc_auc_score(y_test[m_test==20+j],pred_true)
        test_aucs_mouse_notTrue[i,j] = roc_auc_score(y_test[m_test==20+j],
                                                     pred_standard)        
        print(model.Phi,test_aucs_mouse[i,j],test_aucs_mouse_notTrue[i,j])
    print(i,test_aucs[i])

In [ ]:
region_list = ['IL','LHb','LSN','MDThal','MeA','NAc','OFC','PL','V1','VHipp','VMHvl']
for i in range(11):
    print('Region ',region_list[i],test_aucs[i])